# Pipeline Landing para Bronze — CineData Analytics

Este notebook realiza a ingestão dos dados brutos da camada **Landing** (arquivos CSV e API do Banco Central) para a camada **Bronze** em formato Parquet.

### Padrões Técnicos Aplicados:
- **Esquemas Canônicos e Validação**: Definição de `StructType` com validação de conformidade de colunas para cada fonte de dados.
- **Rastreabilidade**: Adição do carimbo de data e hora (`ingestion_datetime`) em todas as tabelas persistidas na Bronze.

In [1]:
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import requests
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Detecção dinâmica de ambiente: Databricks vs. Local
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Resolução de caminhos do Lakehouse
WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = WORKSPACE_DIR / "data"
LANDING_DIR = DATA_DIR / "landing"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

for target_directory in [LANDING_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    target_directory.mkdir(parents=True, exist_ok=True)

# Inicialização ou obtenção da Sessão Spark
if IS_DATABRICKS:
    spark = SparkSession.builder.getOrCreate()
else:
    spark = (
        SparkSession
        .builder
        .appName("CineData_01_Landing_to_Bronze")
        .config("spark.driver.memory", "4g")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.sql.ansi.enabled", "false")
        .getOrCreate()
    )
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.ansi.enabled", "false")

# Provisionamento do banco de dados (database) da camada Bronze
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

# Função helper unificada para exibição rica no Databricks e compatível localmente
def display_dataframe(dataframe: DataFrame, row_limit: int = 10) -> None:
    """
    Exibe o DataFrame utilizando display() nativo no Databricks ou show() em ambiente local.
    """
    display_function = globals().get("display") or getattr(__builtins__, "display", None)
    if callable(display_function):
        display_function(dataframe)
    else:
        dataframe.show(row_limit, truncate=False)

def enforce_dataframe_schema(
    dataframe: DataFrame,
    expected_schema: StructType,
    strict_columns: bool = True
) -> DataFrame:
    """
    Valida a presença de todos os campos definidos no StructType, aplica conversão defensiva
    e reordena as colunas. Em modo estrito, impede colunas não declaradas no contrato.
    """
    actual_column_names = set(dataframe.columns)
    expected_column_names = [field.name for field in expected_schema.fields]
    missing_columns = set(expected_column_names) - actual_column_names

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {missing_columns}")

    if strict_columns:
        unexpected_columns = actual_column_names - set(expected_column_names)
        if unexpected_columns:
            raise ValueError(f"Colunas imprevistas encontradas no DataFrame: {unexpected_columns}")

    ordered_column_expressions = [
        col(field.name).cast(field.dataType).alias(field.name)
        for field in expected_schema.fields
    ]
    return dataframe.select(ordered_column_expressions)

def write_dataframe_with_timestamp(
    dataframe: DataFrame,
    target_path: Path,
    table_name: str | None = None,
    storage_format: str = "delta" if IS_DATABRICKS else "parquet",
    save_mode: str = "append"
) -> None:
    """
    Persiste o DataFrame com a coluna ingestion_datetime no formato Delta (modo append)
    registrando no catálogo gerenciado e preservando persistência física.
    """
    dataframe_with_timestamp = dataframe.withColumn("ingestion_datetime", current_timestamp())
    writer = (
        dataframe_with_timestamp.write
        .format(storage_format)
        .mode(save_mode)
    )
    if IS_DATABRICKS and table_name:
        writer.saveAsTable(table_name)
    else:
        writer.save(str(target_path))

def validate_landing_zone_artifacts(
    landing_path: Path,
    expected_filenames: list[str]
) -> bool:
    """
    Verifica a presença física e integridade de tamanho dos arquivos na Landing Zone.
    """
    existing_files = {
        file_entry.name: file_entry.stat().st_size
        for file_entry in landing_path.glob("*.csv")
    }
    missing_files = [filename for filename in expected_filenames if filename not in existing_files]

    if missing_files:
        print(f"[ALERTA CRÍTICO] Arquivos ausentes na Landing Zone ({landing_path}):")
        for missing_filename in missing_files:
            print(f"  - ❌ {missing_filename}")
        return False

    print(f"[SUCESSO] Todos os {len(expected_filenames)} arquivos de entrada estão disponíveis:")
    for filename in expected_filenames:
        file_size_kb = existing_files[filename] / 1024
        print(f"  - ✅ {filename} ({file_size_kb:.1f} KB)")
    return True


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 23:52:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/19 23:52:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/19 23:52:39 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/19 23:52:39 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


## 1. Ingestão de Arquivos CSV da Camada Landing

Leitura e conformação de esquemas para cada conjunto de dados brutos antes da persistência em formato Parquet na camada Bronze.
* A coluna `tconst` está no dataset `movies_info_TMDB_IMDB` não foi considerada por não aparecer no contrato

In [2]:
LANDING_INGESTION_SPECS = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}

# Ingestão dos arquivos brutos CSV para a camada Bronze em formato Delta (modo append),
# preservando os dados no estado original (todas as colunas como string)
# e carimbando a coluna de rastreabilidade temporal 'ingestion_datetime'.
for source_filename, bronze_table_name in LANDING_INGESTION_SPECS.items():
    raw_dataframe = spark.read.csv(str(LANDING_DIR / source_filename), header=True, inferSchema=False)
    write_dataframe_with_timestamp(
        dataframe=raw_dataframe,
        target_path=BRONZE_DIR / bronze_table_name,
        table_name=bronze_table_name,
        storage_format="delta" if IS_DATABRICKS else "parquet",
        save_mode="append"
    )
    print(f"Ingestão concluída com sucesso: {bronze_table_name}")


Ingestão concluída com sucesso: bronze.tb_movies_info


Ingestão concluída com sucesso: bronze.tb_movies_financials


Ingestão concluída com sucesso: bronze.tb_movies_metrics


Ingestão concluída com sucesso: bronze.tb_credits_and_tags


Ingestão concluída com sucesso: bronze.tb_movies_reviews


## 2. Ingestão da API do Banco Central (Cotação do Dólar)

Consulta à API PTAX Olinda do BACEN para obtenção do histórico de cotações de compra do Dólar americano.

In [3]:
from pydantic import ConfigDict, Field
from sparkdantic import SparkModel


class CotacaoItem(SparkModel):
    model_config = ConfigDict(populate_by_name=True)
    
    cotacao_compra: float = Field(alias="cotacaoCompra")
    data_hora_cotacao: datetime = Field(alias="dataHoraCotacao")

class PtaxResponse(SparkModel):
    model_config = ConfigDict(populate_by_name=True)

    odata_context: str = Field(alias="@odata.context")
    value: list[CotacaoItem]

In [4]:
# Formato esperado de data pela API PTAX Olinda: MM-DD-AAAA.
# Suporte a parâmetros (widgets) no Databricks com fallback automatizado para os últimos 7 dias.
BACEN_API_DATE_FORMAT = "%m-%d-%Y"
QUERY_INTERVAL_DAYS = 7
RECIFE_TIMEZONE = "America/Recife"
DOLAR_QUOTE_ENDPOINT = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    "@dataInicial='{start_date}'&@dataFinalCotacao='{end_date}'&=dataHoraCotacao,cotacaoCompra&=json"
)
BacenCotacaoBronzeSchema = StructType([
    StructField("cotacaoCompra", DoubleType(), nullable=True),
    StructField("dataHoraCotacao", StringType(), nullable=True),
])

current_datetime_recife = datetime.now(ZoneInfo(RECIFE_TIMEZONE))
calculated_default_end = current_datetime_recife.strftime(BACEN_API_DATE_FORMAT)
calculated_default_start = (current_datetime_recife - timedelta(days=QUERY_INTERVAL_DAYS)).strftime(BACEN_API_DATE_FORMAT)

# Configuração e leitura de widgets no Databricks
if "dbutils" in globals():
    try:
        dbutils.widgets.text("data_inicio", calculated_default_start, "Data Início (MM-DD-AAAA)")
        dbutils.widgets.text("data_fim", calculated_default_end, "Data Fim (MM-DD-AAAA)")
        start_date_param = dbutils.widgets.get("data_inicio") or calculated_default_start
        end_date_param = dbutils.widgets.get("data_fim") or calculated_default_end
    except Exception:
        start_date_param = calculated_default_start
        end_date_param = calculated_default_end
else:
    start_date_param = calculated_default_start
    end_date_param = calculated_default_end

response = requests.get(
    DOLAR_QUOTE_ENDPOINT.format(
        start_date=start_date_param,
        end_date=end_date_param
    )
)
response.raise_for_status()
validated_response = PtaxResponse.model_validate(response.json())
cotacao_records = [cotacao_item.model_dump(by_alias=True) for cotacao_item in validated_response.value]

# Persistência dos dados de cotação na Bronze com esquema validado e carimbo de ingestão
dataframe_cotacao_raw = spark.createDataFrame(data=cotacao_records, schema=BacenCotacaoBronzeSchema)
dataframe_cotacao_validated = enforce_dataframe_schema(dataframe_cotacao_raw, BacenCotacaoBronzeSchema)
write_dataframe_with_timestamp(
    dataframe=dataframe_cotacao_validated,
    target_path=BRONZE_DIR / "bronze.tb_cotacao_dolar",
    table_name="bronze.tb_cotacao_dolar",
    storage_format="delta" if IS_DATABRICKS else "parquet",
    save_mode="append"
)
display_dataframe(dataframe_cotacao_validated, 10)


DataFrame[cotacaoCompra: double, dataHoraCotacao: string]